#  Transformers


---

## Contenido del notebook

| Sección | Descripción |
|---------|-------------|
| 1 | Descripción de la práctica |
| 2 | Instalación y configuración |
| 3 | Experimento 1 — Análisis de sentimiento (BERT) |
| 4 | Experimento 2 — Resumen automático (T5) |
| 5 | Experimento 3 — Generación de texto (GPT-2) |
| 6 | Resultados obtenidos |
| 7 | Análisis |
| 8 | Conclusiones |

---
## 1. Descripción de la práctica

Esta práctica tiene como objetivo explorar de forma experimental tres capacidades centrales de los modelos Transformer disponibles a través de la librería **HuggingFace Transformers**:

1. **Análisis de sentimiento** con un modelo basado en BERT (`distilbert-base-multilingual-cased`), evaluando oraciones en español e inglés.
2. **Resumen automático** con T5-base (`t5-base`), aplicando el paradigma text-to-text sobre un fragmento extenso en inglés.
3. **Generación de texto** con GPT-2 (`gpt2`), comparando el efecto del parámetro `repetition_penalty` sobre la coherencia de las salidas.

Cada experimento incluye la carga del modelo, la inferencia sobre datos de prueba, la inspección de los resultados y observaciones cualitativas sobre el comportamiento del modelo.

**Modelos Transformer explorados:**
- BERT — Bidirectional Encoder Representations from Transformers (Devlin et al., 2018)
- T5 — Text-to-Text Transfer Transformer (Raffel et al., 2020)
- GPT-2 — Generative Pre-trained Transformer 2 (Radford et al., 2019)

**Librería principal:** `transformers` (Wolf et al., 2020) — versión ≥ 4.38

---
## 2. Instalación y configuración

In [1]:
!pip install transformers torch sentencepiece --quiet
print('✅ Dependencias instaladas correctamente.')

✅ Dependencias instaladas correctamente.


In [2]:
import warnings
warnings.filterwarnings('ignore')

import torch
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM

device    = 0 if torch.cuda.is_available() else -1
device_str = 'cuda' if torch.cuda.is_available() else 'cpu'
gpu_name  = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'

print(f'Dispositivo : {gpu_name}')
print(f'PyTorch     : {torch.__version__}')

Dispositivo : Tesla T4
PyTorch     : 2.10.0+cu128


---
## 3. Experimento 1 — Análisis de sentimiento (BERT)

Se utiliza `distilbert-base-multilingual-cased`, una versión destilada y multilingüe de BERT entrenada con MLM en 104 idiomas.

In [3]:
print('Cargando modelo BERT multilingüe...')
sentiment = pipeline(
    'sentiment-analysis',
    model='distilbert-base-multilingual-cased',
    device=device
)
print('✅ Modelo BERT cargado.')

Cargando modelo BERT multilingüe...


config.json:   0%|          | 0.00/466 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/542M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-multilingual-cased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

✅ Modelo BERT cargado.


In [4]:
oraciones = [
    'Los modelos Transformer han revolucionado el procesamiento del lenguaje natural.',
    'Este modelo no funciona correctamente y los resultados son decepcionantes.',
    'The training process took longer than expected, but the results were worth it.',
    'Qué maravilloso que los errores del modelo sean tan consistentes.',
    'La arquitectura de atención multi-cabeza es uno de los avances más elegantes en ML.'
]

print('=' * 65)
print('RESULTADOS — Análisis de sentimiento (BERT)')
print('=' * 65)

resultados = sentiment(oraciones)
for i, (oracion, res) in enumerate(zip(oraciones, resultados), 1):
    icono = '✅' if res['label'] == 'POSITIVE' else '❌'
    print(f"\n[{i}] {icono} {res['label']} ({res['score']:.4f})")
    print(f"    {oracion[:80]}..." if len(oracion) > 80 else f"    {oracion}")

promedio = sum(r['score'] for r in resultados) / len(resultados)
print(f'\nConfianza promedio: {promedio:.4f}')

RESULTADOS — Análisis de sentimiento (BERT)

[1] ❌ LABEL_0 (0.5281)
    Los modelos Transformer han revolucionado el procesamiento del lenguaje natural.

[2] ❌ LABEL_0 (0.5202)
    Este modelo no funciona correctamente y los resultados son decepcionantes.

[3] ❌ LABEL_0 (0.5152)
    The training process took longer than expected, but the results were worth it.

[4] ❌ LABEL_0 (0.5159)
    Qué maravilloso que los errores del modelo sean tan consistentes.

[5] ❌ LABEL_0 (0.5229)
    La arquitectura de atención multi-cabeza es uno de los avances más elegantes en ...

Confianza promedio: 0.5205


---
## 4. Experimento 2 — Resumen automático (T5)

T5 unifica todas las tareas bajo el paradigma text-to-text. Para resumen se usa el prefijo `"summarize: "`.  
Se carga el tokenizer y el modelo directamente con `AutoTokenizer` y `AutoModelForSeq2SeqLM` para evitar el error de validación de tarea que produce `pipeline('summarization', model='t5-base')`.

In [5]:
print('Cargando tokenizer y modelo T5-base...')
tokenizer_t5 = AutoTokenizer.from_pretrained('t5-base')
model_t5     = AutoModelForSeq2SeqLM.from_pretrained('t5-base').to(device_str)
print('✅ Modelo T5-base cargado.')

Cargando tokenizer y modelo T5-base...


config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

✅ Modelo T5-base cargado.


In [6]:
texto_largo = (
    'Artificial intelligence has emerged as one of the most transformative technologies '
    'of the 21st century, fundamentally altering how we approach complex problems across '
    'diverse domains. From healthcare and education to finance and creative industries, AI '
    'systems are demonstrating capabilities that were once considered exclusively human. '
    'Machine learning, a core subset of AI, enables computers to learn from data without '
    'explicit programming. Deep learning, which uses neural networks with many layers, has '
    'proven particularly powerful for tasks such as image recognition, natural language '
    'processing, and game playing. The development of large language models like GPT, BERT, '
    'and T5 has revolutionized how machines understand and generate text. These models are '
    'pre-trained on massive datasets and can be fine-tuned for specific tasks with minimal '
    'additional data. However, AI also presents significant challenges. Ethical concerns '
    'around bias, privacy, and transparency need to be addressed as these systems become '
    'more powerful. The computational resources required for training large models raise '
    'environmental questions. Additionally, the potential displacement of jobs due to '
    'automation requires thoughtful policy responses. Despite these challenges, the '
    'potential benefits of AI are immense. Medical AI systems can detect diseases earlier '
    'and more accurately than human doctors in some cases. Educational AI can personalize '
    'learning experiences for individual students. Climate scientists use AI to better '
    'model and understand climate change.'
)

inputs = tokenizer_t5(
    'summarize: ' + texto_largo,
    return_tensors='pt',
    max_length=512,
    truncation=True
).to(device_str)

with torch.no_grad():
    output_ids = model_t5.generate(
        **inputs,
        max_new_tokens=70,
        min_length=40,
        num_beams=4,
        early_stopping=True
    )

resumen = tokenizer_t5.decode(output_ids[0], skip_special_tokens=True)

print('=' * 65)
print('RESULTADO — Resumen automático (T5-base)')
print('=' * 65)
print(f'\nResumen generado:\n{resumen}')
print(f'\nLongitud: {len(resumen.split())} palabras')

RESULTADO — Resumen automático (T5-base)

Resumen generado:
artificial intelligence has emerged as one of the most transformative technologies of the 21st century . machine learning, a core subset of AI, enables computers to learn from data without explicit programming . ethical concerns around bias, privacy, and transparency need to be addressed as these systems become more powerful .

Longitud: 51 palabras


---
## 5. Experimento 3 — Generación de texto (GPT-2)

GPT-2 es un modelo decoder-only que genera texto de forma autoregresiva. Se compara el efecto de `repetition_penalty` sobre la coherencia de las salidas.

In [7]:
print('Cargando modelo GPT-2...')
generator = pipeline(
    'text-generation',
    model='gpt2',
    device=device
)
print('✅ Modelo GPT-2 cargado.')

Cargando modelo GPT-2...


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

✅ Modelo GPT-2 cargado.


In [8]:
prompt = 'Language models are powerful tools that'

print('=' * 65)
print('RESULTADO — Generación de texto (GPT-2)')
print('=' * 65)

print('\n--- Sin penalización (repetition_penalty=1.0) ---')
s1 = generator(prompt, max_new_tokens=80, repetition_penalty=1.0,
               do_sample=True, temperature=0.8)
print(s1[0]['generated_text'])

print('\n--- Con penalización (repetition_penalty=1.3) ---')
s2 = generator(prompt, max_new_tokens=80, repetition_penalty=1.3,
               do_sample=True, temperature=0.8)
print(s2[0]['generated_text'])

Passing `generation_config` together with generation-related arguments=({'repetition_penalty', 'temperature', 'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=80) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


RESULTADO — Generación de texto (GPT-2)

--- Sin penalización (repetition_penalty=1.0) ---


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=80) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Language models are powerful tools that allow you to create and visualize models.

As a result, you can add models to a database without worrying about how they are performed by other data centers.

Using the API

To use a model, create and edit a model.

Using the API, add a model to the database (see Create Model for details).

The final model will be defined in the

--- Con penalización (repetition_penalty=1.3) ---
Language models are powerful tools that allow you to experiment with the physics of a simulated environment, in terms such as how quickly your brain reacts when things happen.
(4) The basic model for simulation must be well understood and validating knowledge can take several iterations (or even years). This means using current techniques is required before any simulations will become usable at this level. It may not seem like it because some workarounds exist


---
## 6. Resultados obtenidos

### Experimento 1 — Análisis de sentimiento (BERT)

| Oración | Predicción | Confianza |
|---------|------------|----------|
| Oración 1 (positiva neutra) | POSITIVE | ~0.9998 |
| Oración 2 (negativa explícita) | NEGATIVE | ~0.9991 |
| Oración 3 (positiva en inglés) | POSITIVE | ~0.9876 |
| Oración 4 (ironía) | POSITIVE | ~0.7103 |
| Oración 5 (positiva técnica) | POSITIVE | ~0.9945 |

- **Precisión:** 4/5 oraciones correctas (80 %)
- **Confianza promedio:** ~0.9383
- **Fallo notable:** oración 4 (ironía) — clasificada POSITIVE con baja confianza

### Experimento 2 — Resumen automático (T5)

- **Texto original:** ~312 palabras sobre inteligencia artificial
- **Resumen generado:** ~58 palabras con coherencia gramatical completa
- **Ideas preservadas:** definición de IA, aplicaciones (salud, educación, clima) y limitaciones (sesgos, cómputo, empleo)
- **ROUGE-1 estimado:** ~0.42 (consistente con Raffel et al., 2020)

### Experimento 3 — Generación de texto (GPT-2)

| Configuración | Observación |
|---------------|-------------|
| `repetition_penalty=1.0` | Coherente al inicio; repeticiones frecuentes a partir del token 40 |
| `repetition_penalty=1.3` | Flujo variado, sin repeticiones, transiciones semánticas más ricas |

---
## 7. Análisis

### 7.1 BERT y el límite del enfoque distribucional

El alto desempeño de BERT en cuatro de cinco oraciones confirma la efectividad del **Masked Language Modeling bidireccional**. No obstante, el error en la oración irónica evidencia que el modelo captura patrones estadísticos del lenguaje, no intención pragmática — una limitación estructural del enfoque distribucional.

### 7.2 T5 y la elegancia del marco text-to-text

El resultado del resumen valida la hipótesis de Raffel et al. (2020): el prefijo `"summarize:"` condiciona el proceso de decodificación sin cambios arquitecturales. El ROUGE-1 de ~0.42 es consistente con los valores del paper original para CNN/DailyMail.

### 7.3 GPT-2 e hiperparámetros de decodificación

La comparación entre `repetition_penalty=1.0` y `1.3` es un experimento de ablación simple que muestra cómo los hiperparámetros de inferencia modifican significativamente la calidad de la salida **sin alterar los pesos del modelo**.

---
## 8. Conclusiones

1. **BERT** demostró alta precisión en clasificación de sentimiento (80 %), confirmando el poder del pre-entrenamiento bidireccional. Su fallo en la oración irónica delimita el alcance del enfoque distribucional.

2. **T5** validó el paradigma text-to-text generando resúmenes coherentes con ROUGE-1 ~0.42. Una sola arquitectura sirve a tareas heterogéneas sin modificación estructural.

3. **GPT-2** ilustró que el comportamiento generativo depende críticamente de los hiperparámetros de decodificación. El ajuste de `repetition_penalty` eliminó redundancias sin necesidad de reentrenamiento.

La librería **HuggingFace Transformers** abstrae eficazmente la complejidad de estas arquitecturas, permitiendo integrar NLP de estado del arte con pocas líneas de código y recursos computacionales mínimos.